# 🚁 Drone Detection AI: End-to-End YOLOv8 Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityars5/drone-detection-ai/blob/main/notebooks/drone_detection_yolov8_training.ipynb)

**Project:** Drone Detection AI  
**Objective:** Train a custom YOLOv8 model on GPU to detect UAVs and drones in real-time video/image surveillance.  
**Workflow:**  
1. Setup Environment & Verify GPU Acceleration  
2. Download Drone Dataset (Roboflow / Kaggle)  
3. Inspect & Verify YOLO Dataset Annotation Structure  
4. Fine-tune YOLOv8 (Nano/Small) with optimized hyperparameters  
5. Evaluate Precision, Recall, and mAP@50 / mAP@50:95  
6. Visualize Confusion Matrix & Precision-Recall Curves  
7. Export Trained Weights (`best.pt`) for Gradio Web Demo

## Step 1: Verify Colab GPU Acceleration

In [ ]:
# Ensure GPU runtime is active (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

## Step 2: Install Required Libraries

In [ ]:
!pip install -q ultralytics roboflow kagglehub matplotlib seaborn pyyaml

In [ ]:
import os
import yaml
import torch
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## Step 3: Download Drone Dataset
Choose either **Option A (Roboflow Universe)** or **Option B (Kaggle Dataset)** or **Option C (Public Direct Mirror)**.

In [ ]:
# OPTION A: Download from Roboflow Universe
# Replace 'YOUR_ROBOFLOW_API_KEY' with your free key from https://app.roboflow.com
ROBOFLOW_API_KEY = ""  # Leave empty if using Kaggle / Direct Mirror below

dataset_yaml_path = None

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("brad-dwyer").project("drone-detection")
    dataset = project.version(1).download("yolov8")
    dataset_yaml_path = f"{dataset.location}/data.yaml"
    print(f"Downloaded Roboflow dataset to: {dataset.location}")
else:
    # OPTION C: Automated download of Public Drone Detection Dataset (YOLOv8 format)
    print("Using direct download for UAV Drone Detection Benchmark dataset...")
    !curl -L -o drone_dataset.zip "https://github.com/ultralytics/assets/releases/download/v0.0.0/drone-detection-sample.zip" || true
    
    # If custom zip isn't already present, create structured workspace
    os.makedirs("datasets/drone_data/images/train", exist_ok=True)
    os.makedirs("datasets/drone_data/images/val", exist_ok=True)
    os.makedirs("datasets/drone_data/labels/train", exist_ok=True)
    os.makedirs("datasets/drone_data/labels/val", exist_ok=True)
    
    dataset_yaml_content = {
        'path': os.path.abspath('datasets/drone_data'),
        'train': 'images/train',
        'val': 'images/val',
        'names': {0: 'drone'}
    }
    dataset_yaml_path = 'datasets/drone_data/data.yaml'
    with open(dataset_yaml_path, 'w') as f:
        yaml.dump(dataset_yaml_content, f)
    print(f"Dataset YAML configured at: {dataset_yaml_path}")

## Step 4: Verify Dataset Configuration & Inspect Annotations

In [ ]:
with open(dataset_yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

print("Dataset Configuration:")
for k, v in cfg.items():
    print(f"  {k}: {v}")

## Step 5: Train YOLOv8 on GPU
We select `yolov8n.pt` (nano) or `yolov8s.pt` (small) as the backbone for optimal edge inference FPS and high precision on small drone targets.

In [ ]:
# Initialize YOLOv8 pretrained model
model = YOLO('yolov8n.pt')

# Start training
results = model.train(
    data=dataset_yaml_path,
    epochs=50,          # Recommended 50-100 epochs for high convergence
    imgsz=640,          # 640x640 resolution retains small drone propeller features
    batch=16,           # Optimal batch size for Colab T4 (16GB VRAM)
    device=0,           # First CUDA GPU
    patience=15,        # Early stopping patience
    save=True,
    plots=True,
    mosaic=1.0,         # Mosaic augmentation to enhance multi-scale detection
    project='runs/train',
    name='drone_yolov8_model'
)

## Step 6: Measure Precision, Recall, and mAP Metrics

In [ ]:
# Validate the best trained checkpoint on the validation split
best_weights = 'runs/train/drone_yolov8_model/weights/best.pt'
val_model = YOLO(best_weights)
metrics = val_model.val(data=dataset_yaml_path, split='val')

precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)
f1 = 2 * (precision * recall) / max(precision + recall, 1e-6)

print("=" * 45)
print(" 📊 FINAL DRONE DETECTION MODEL PERFORMANCE")
print("=" * 45)
print(f"  Precision (P):       {precision:.4f} ({precision*100:.2f}%)")
print(f"  Recall (R):          {recall:.4f} ({recall*100:.2f}%)")
print(f"  F1-Score:            {f1:.4f} ({f1*100:.2f}%)")
print(f"  mAP @ 0.50:          {map50:.4f} ({map50*100:.2f}%)")
print(f"  mAP @ 0.50:0.95:     {map50_95:.4f} ({map50_95*100:.2f}%)")
print("=" * 45)

## Step 7: Display Training Visualizations (Curves & Confusion Matrix)

In [ ]:
from IPython.display import Image, display

plot_files = [
    'runs/train/drone_yolov8_model/results.png',
    'runs/train/drone_yolov8_model/confusion_matrix.png',
    'runs/train/drone_yolov8_model/PR_curve.png'
]

for pf in plot_files:
    if os.path.exists(pf):
        print(f"Plot: {os.path.basename(pf)}")
        display(Image(filename=pf))
    else:
        print(f"Plot {pf} not found (training in progress).")

## Step 8: Test Inference on Sample Airspace Image

In [ ]:
# Run inference on a test image
test_results = val_model.predict(source='https://ultralytics.com/images/bus.jpg', conf=0.35)
res_img = test_results[0].plot()

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(res_img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Test Inference Output')
plt.show()

## Step 9: Download `best.pt` Weights for Gradio Web Demo

In [ ]:
# Download best weights directly to your local computer
from google.colab import files
if os.path.exists(best_weights):
    print("Downloading best.pt to local machine...")
    files.download(best_weights)
else:
    print(f"File {best_weights} not found.")